# M06B: Multiple Functions

Real applications use dozens of functions.  

The model must choose the right one (routing), chain them (sequential), or run them together (parallel).

**Topics:**
- Defining multiple functions and intelligent routing
- Sequential vs. parallel execution patterns
- Multi-function assistant with routing and execution handling

---

## 🔧 Step 1: Setup

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"

print(f"✅ Setup complete: Using {MODEL}!")

---

## 🏗️ Part 1: Defining Multiple Functions

Define three different functions to test intelligent routing.

In [ ]:
# Define multiple functions
def get_weather(location):
    """Get weather for location."""
    # Simulated weather data (temperatures in °C)
    weather = {
        "san francisco": {"temp": 22, "condition": "sunny"},
        "new york": {"temp": 18, "condition": "cloudy"},
        "london": {"temp": 14, "condition": "rainy"}
    }
    return weather.get(location.strip().lower(), {"temp": None, "condition": "unknown"})

def get_user_info(user_id):
    """Get user information."""
    user_id = str(user_id)  # Handle numeric IDs from model
    users = {
        "123": {"name": "Alice", "email": "alice@example.com", "plan": "premium"},
        "456": {"name": "Bob", "email": "bob@example.com", "plan": "free"},
        "789": {"name": "Charlie", "email": "charlie@example.com", "plan": "business"}
    }
    return users.get(user_id, {"error": "User not found"})

def calculate(operation, a, b):
    """Perform calculation."""
    if operation == "divide" and b == 0:
        return {"error": "Division by zero"}

    if operation == "add":
        return {"result": a + b}
    if operation == "subtract":
        return {"result": a - b}
    if operation == "multiply":
        return {"result": a * b}
    if operation == "divide":
        return {"result": a / b}
    
    return {"error": "Unknown operation"}


# --------------------------------------------------------------
print("✅ Functions defined")

### Define the Tool Schemas

Schemas allow the model to compare descriptions and choose the right function.

In [ ]:
# Define tool schemas for all functions (Responses API Format)
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get current weather for a location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City name"}
            },
            "required": ["location"]
        }
    },
    {
        "type": "function",
        "name": "get_user_info",
        "description": "Get user information by ID",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {"type": "string", "description": "User ID"}
            },
            "required": ["user_id"]
        }
    },
    {
        "type": "function",
        "name": "calculate",
        "description": "Perform mathematical calculation",
        "parameters": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"]
                },
                "a": {"type": "number"},
                "b": {"type": "number"}
            },
            "required": ["operation", "a", "b"]
        }
    }
]

# Map function names to implementations
available_functions = {
    "get_weather": get_weather,
    "get_user_info": get_user_info,
    "calculate": calculate
}


# --------------------------------------------------------------
print("✅ Tools schema ready")

---

## 🎯 Part 2: Model Choosing Functions

The model analyzes intent to pick the correct tool. Let's see this intelligent routing in action.

In [ ]:
def run_conversation(user_message, tools, available_functions):
    """Run a function-calling loop. Stops when model returns no tool calls."""
    
    # Each item must be a dict when using a list
    conversation_items = [{"role": "user", "content": user_message}]
    
    # Loop until no tool calls (max 3 rounds to prevent infinite loops)
    for _ in range(3):
        response = client.responses.create(
            model=MODEL,
            input=conversation_items,
            tools=tools
        )
        
        # Filter for function calls only
        tool_items = [
            i for i in response.output
            if i.type == "function_call"
        ]
        
        # No tool calls — return final response
        if not tool_items:
            return (response.output_text or "").strip()
        
        # Process each tool call
        for tool_item in tool_items:
            function_name = tool_item.name
            call_id = tool_item.call_id
            
            try:
                function_args = json.loads(tool_item.arguments)
            except Exception:
                return "Error: could not parse tool arguments."
            
            if function_name not in available_functions:
                return f"Error: unknown tool '{function_name}'."
            
            # Add function call to conversation
            conversation_items.append({
                "type": "function_call",
                "call_id": call_id,
                "name": function_name,
                "arguments": tool_item.arguments
            })
            
            # Execute function and add output
            result = available_functions[function_name](**function_args)
            conversation_items.append({
                "type": "function_call_output",
                "call_id": call_id,
                "output": json.dumps(result)
            })
    
    return "Error: too many tool-call rounds."


# --------------------------------------------------------------
print("✅ Helper function ready")

### Test the Router

In [ ]:
# Test model choosing correct function
print("🎯 MODEL FUNCTION SELECTION")
print("="*60)

questions = [
    "What's the weather in London?",
    "Show me info for user 123",
    "Calculate 45 times 8",
    "What's 100 divided by 4?"
]

for q in questions:
    print(f"\nQ: {q}")
    answer = run_conversation(q, tools, available_functions)
    print(f"A: {answer}")

### How Model Chooses Functions

**Factors:**
- **Intent:** Analyzes user question
- **Schema:** Matches intent to function descriptions
- **Parameters:** Checks if it can fill required args

**Insight:** Clear, distinct descriptions are critical for accurate routing.

---

## 🔗 Part 3: Sequential Function Calls

Sequential calls happen when one function's output is needed as another's input (e.g., Get location → Get weather).

In [ ]:
# Add function that depends on another
def get_user_location(user_id):
    """Get user's location."""
    user_id = str(user_id)  # Handle numeric IDs from model
    locations = {
        "123": "San Francisco",
        "456": "New York",
        "789": "London"
    }
    return {"location": locations.get(user_id, "Unknown")}

# Add to tools (Responses API format)
tools_extended = tools + [
    {
        "type": "function",
        "name": "get_user_location",
        "description": "Get user's city location",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {"type": "string"}
            },
            "required": ["user_id"]
        }
    }
]

available_functions["get_user_location"] = get_user_location


# --------------------------------------------------------------
print("✅ Sequential functions ready")

### Test Sequential Execution

In [ ]:
# Test sequential calls
print("🔗 SEQUENTIAL FUNCTION CALLS")
print("="*60)

# This requires: get_user_location → get_weather
question = "What's the weather where user 123 is located?"
print(f"\nQ: {question}")
answer = run_conversation(question, tools_extended, available_functions)
print(f"\nA: {answer}")

print("="*60)

### Understanding Sequential Patterns

**Insight:** Sequential calls require multiple API rounds — each dependent call adds another round before the final response.

---

## ⚡ Part 4: Parallel Function Calls

When functions are **independent**, the model can call multiple at once in the same turn.

In [ ]:
# Test parallel calls
print("⚡ PARALLEL FUNCTION CALLS")
print("="*60)

question = "What's the weather in London and New York?"
print(f"\nQ: {question}")
answer = run_conversation(question, tools_extended, available_functions)
print(f"\nA: {answer}")

print("="*60)

### Parallel vs Sequential

**Sequential (Multiple Turns):**
- Dependent functions (output feeds input)
- Multiple API requests required

**Parallel (Same Turn):**
- Independent functions
- Single API request for multiple calls
- More efficient

**💡 Insight:** The `run_conversation` helper handles both patterns — the outer loop handles multiple rounds (sequential), the inner loop handles multiple calls per round (parallel).

---

## 🤖 Part 5: Production Assistant Class

Wrap everything into a class with logging and turn limits.

In [ ]:
class FunctionAssistant:
    """Production assistant with multiple functions."""
    
    def __init__(self, client, model, tools, functions, instructions=None):
        self.client = client
        self.model = model
        self.tools = tools
        self.functions = functions
        self.instructions = instructions
        self.call_log = []
    
    def chat(self, user_message, max_turns=5):
        """Chat with function calling support."""
        # Same loop pattern as run_conversation, with logging added
        conversation_items = [{"role": "user", "content": user_message}]
        
        for _ in range(max_turns):
            response = self.client.responses.create(
                model=self.model,
                input=conversation_items,
                tools=self.tools,
                instructions=self.instructions
            )
            
            # Filter for function calls
            tool_items = [
                i for i in response.output
                if i.type == "function_call"
            ]
            
            # No tool calls — return final response
            if not tool_items:
                return response.output_text
            
            for item in tool_items:
                function_name = item.name
                function_args = json.loads(item.arguments)
                
                # Log call
                self.call_log.append({"function": function_name, "args": function_args})
                
                # Add function call to conversation
                conversation_items.append({
                    "type": "function_call",
                    "call_id": item.call_id,
                    "name": function_name,
                    "arguments": item.arguments
                })
                
                # Execute and add output
                result = self.functions[function_name](**function_args)
                conversation_items.append({
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps(result)
                })
        
        return "Error: too many tool-call rounds."
    
    def get_stats(self):
        """Get call statistics."""
        return {
            "total_calls": len(self.call_log),
            "functions_used": sorted(set(c["function"] for c in self.call_log))
        }


# --------------------------------------------------------------
print("✅ FunctionAssistant ready")

### Test the Assistant

In [ ]:
# Test production assistant
print("🤖 PRODUCTION ASSISTANT")
print("="*60)

assistant = FunctionAssistant(
    client=client,
    model=MODEL,
    tools=tools_extended,
    functions=available_functions
)

queries = [
    "What's the weather where user 456 lives?",
    "What's the weather in London and New York?",
    "Calculate 15 times 8, then show me user 123's info"
]

for q in queries:
    print(f"\nQ: {q}")
    answer = assistant.chat(q)
    print(f"A: {answer}")

print("\n" + "="*60)
print("STATISTICS")
stats = assistant.get_stats()
print(f"Total function calls: {stats['total_calls']}")
print(f"Functions used: {', '.join(stats['functions_used'])}")

---

### 💪 Your Turn: Build Your Own

Build a multi-function order management system with all 4 functions:
1. `get_order(order_id)` - Get order details
2. `get_shipping_status(order_id)` - Check shipping
3. `cancel_order(order_id)` - Cancel order
4. `get_customer(customer_id)` - Customer info

Test with `run_conversation` to verify routing and sequential calls.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Order Management System
# --------------------------------------------------------------
# Objective: Build a multi-function order management system.

def get_order(order_id):
    """Get order details."""
    # TODO: Create orders dictionary with sample data
    # orders = {"789": {...}, "456": {...}}
    # TODO: Return order or error if not found
    pass

def get_shipping_status(order_id):
    """Check shipping status."""
    # TODO: Create shipping dictionary
    # TODO: Return shipping info or error
    pass

def cancel_order(order_id):
    """Cancel an order."""
    # TODO: Update order status to "cancelled"
    # TODO: Return confirmation or error
    pass

def get_customer(customer_id):
    """Get customer information."""
    # TODO: Create customers dictionary
    # TODO: Return customer data or error
    pass

# Define your schemas (Responses API format)
order_tools = [
    # TODO: Add schema for get_order
    # {
    #      "type": "function",
    #      "name": "get_order",
    #      "description": "...",
    #      "parameters": {...}
    # },
    # TODO: Add schemas for other 3 functions
]

# Map functions
order_functions = {
    # TODO: Map all function names to implementations
    # "get_order": get_order,
    # ...
}

# Test queries
test_queries = [
    "What's the status of order 789?",
    "Show me all info for customer 123",
    "Cancel order 456 and check its shipping status"
]

# TODO: Test with run_conversation
# for q in test_queries:
#     print(f"\nQ: {q}")
#     answer = run_conversation(q, order_tools, order_functions)
#     print(f"A: {answer}")

print("💡 Build your order management system using the TODOs as a guide!")

## 🎯 Key Takeaways

**Routing:**
- Define each function with a clear, distinct description
- Map function names to implementations in a registry dict
- Model chooses based on user intent + descriptions

**Sequential Calls:**
- Use when one function's output feeds another
- Requires multiple API rounds (3+ for two dependent functions)
- Set `max_turns` high enough to complete the chain

**Parallel Calls:**
- Model requests multiple functions in a single response
- Process all calls, then submit all results together
- Works when functions are independent of each other

**FunctionAssistant Class:**
- Wraps the same loop pattern as `run_conversation`
- Adds call logging for debugging and monitoring
- Set turn limits to prevent infinite loops

---

### 📍 Next Step

**M06C: Capstone #3 — Multi-Tool Agent** — Connect to real APIs with weather, database, and error handling patterns.

---

## 🔧 Troubleshooting

**Model choosing wrong function?**
- Make function descriptions distinct
- Add examples in descriptions

**"Expected an input item, but got a string"?**
- When `input` is a list, each item must be a dict
- Use `{"role": "user", "content": message}` not a bare string

**Functions not parallel?**
- Ensure functions are independent
- Try explicit "compare X and Y" questions

**Sequential not working?**
- Verify max_turns is high enough
- Ensure second function uses first's output

**Missing calls?**
- Check `item.type == "function_call"`
- Verify `response.output` is not empty

**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output

---